# Lab 2: Alternative Ways to Trace

## Difficulty: Beginner | ~35 min | Requires Lab 1

---

### What is wrap_openai?

`wrap_openai` is a function from the LangSmith SDK that wraps an OpenAI client so every API call is automatically traced. Instead of decorating individual functions, you wrap the client once and all calls through it appear in LangSmith.

### What is the trace() Context Manager?

The `trace()` context manager gives you manual control over tracing. You define exactly which operations go into a trace by wrapping them in a `with` block — useful when you need to group multiple unrelated calls or add custom metadata.

### What is LangChain's Callback Tracer?

LangChain has a built-in callback system that automatically traces chains, agents, and retrievers. When you use `@traceable` on chain functions, LangChain records the full execution tree — including tool calls, retriever lookups, and nested LLM calls — without any extra setup.

In [1]:
!pip install -qU langsmith>=0.1.0 openai>=1.0.0 langchain>=0.2.0 langchain-core>=0.2.0 langchain-openai>=0.1.0 python-dotenv>=1.0.0

zsh:1: 0.1.0 not found


This installs the exact versions of every library used in this lab.

## Cell 2: Load Environment and Initialize OpenAI Client

Loads API keys and initializes the OpenAI client pointing to OpenRouter.

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load API keys from .env file
load_dotenv()
assert os.getenv("OPENROUTER_API_KEY"), "Missing OPENROUTER_API_KEY"
assert os.getenv("LANGSMITH_API_KEY"), "Missing LANGSMITH_API_KEY"

# Initialize OpenAI client pointing to OpenRouter (OpenAI-compatible)
openai_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)
print("✓ Client ready")

✓ Client ready


## Cell 3: Method 1 — wrap_openai

`wrap_openai` wraps the client so every API call is automatically traced. No decorator needed.

In [3]:
from langsmith.wrappers import wrap_openai

# Wrap the client — every API call through it is now traced automatically
wrapped_client = wrap_openai(openai_client)

# Make a call through the wrapped client — no @traceable decorator needed
response = wrapped_client.chat.completions.create(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    messages=[{"role": "user", "content": "What is 2+2? Reply with just the number."}]
)
print(f"Response: {response.choices[0].message.content}")

Response: 4


## Cell 4: Method 2 — trace() Context Manager

The `trace()` context manager wraps multiple operations into a single trace. You control exactly what's included and can add custom metadata.

In [4]:
from langsmith import trace

# The trace() context manager groups everything inside the with block
# metadata lets you attach custom tags like method
with trace("manual_grouped_operations", metadata={"method": "context_manager"}) as ts:
    prompt = "What is the capital of France?"
    response = openai_client.chat.completions.create(
        model="nvidia/nemotron-3-super-120b-a12b:free",
        messages=[{"role": "user", "content": prompt}]
    )
    answer = response.choices[0].message.content

print(f"Response: {answer}")

Response: The capital of France is **Paris**.  

This is a well-established fact in geography and has been consistently recognized for centuries. If you'd like additional context (e.g., historical details, population, or cultural significance), feel free to ask! 😊


## Cell 5: Method 3 — LangChain Callback Tracer

LangChain's callback system automatically traces the LLM call and any tool invocations — no manual setup needed.

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

# Define a tool — @tool decorator converts the function into a LangChain tool
@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

# ChatOpenAI with LangChain automatically traces via callbacks
llm = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

# bind_tools attaches the tool — the LLM can decide to call it
llm_with_tools = llm.bind_tools([add])
response = llm_with_tools.invoke([HumanMessage(content="What is 3 + 5?")])

# The LLM may return empty content but include tool_calls — extract the result
if response.tool_calls:
    tc = response.tool_calls[0]
    result = add.invoke(tc["args"])
    print(f"Tool called: {tc['name']}({tc['args']}) = {result}")
else:
    print(f"Response: {response.content}")

Tool called: add({'a': 3, 'b': 5}) = 8


## Cell 6: Inspect Traces

Queries LangSmith to verify all traces were recorded successfully.

In [6]:
from langsmith import Client

ls_client = Client()
traces = list(ls_client.list_runs(
    project_name=os.getenv("LANGSMITH_PROJECT"),
    is_root=True,
    limit=10
))

print(f"✓ Found {len(traces)} trace(s)")
for t in traces:
    print(f"  - {t.name} ({t.run_type}) - {t.status}")

✓ Found 10 trace(s)
  - manual_grouped_operations (chain) - success
  - ChatOpenAI (llm) - success
  - add (tool) - success
  - ChatOpenAI (llm) - success
  - ChatOpenAI (llm) - success
  - ChatOpenAI (llm) - success
  - manual_grouped_operations (chain) - success
  - ChatOpenAI (llm) - success
  - ChatOpenAI (llm) - success
  - manual_grouped_operations (chain) - success
